# 🚀 Cortex CEM - Model Training & Fine-Tuning on Google Colab

This notebook contains the complete workflow to train your Cortex CEM agentic models using **AgentScope** and **Trinity-RFT** with GPU acceleration.

## 🛠️ Step 1: Verify GPU Access
Make sure your Colab instance is using a GPU runtime (**Runtime > Change runtime type > T4 GPU or L4 GPU**).

In [ ]:
!nvidia-smi

## 📦 Step 2: Clone Repository & Install Dependencies
Clone your Cortex repository from GitHub and install all required libraries.

In [ ]:
# Clone the repository
!git clone https://github.com/vikasvardhanv/cortex.git
%cd cortex

# Install dependencies in editable mode
!pip install -e .
!pip install agentscope trinity-rft mlflow huggingface_hub

## 💾 Step 3: Mount Google Drive (Optional but Recommended)
Colab runtimes are temporary. Mount Google Drive to persist training checkpoints so you don't lose progress if your session disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# You can copy or link checkpoints to Google Drive here, e.g.:
# !mkdir -p /content/drive/MyDrive/cortex_checkpoints

## 📈 Step 4: Run Training
Start training the math agent on the GSM8K dataset. You can also specify custom tasks and datasets here.

In [ ]:
# Run the math_agent training task
!python tuner/cli.py math_agent --dataset openai/gsm8k

## 🤗 Step 5: Save & Push Fine-Tuned Model to Hugging Face Hub
Once training completes, you can upload the model checkpoints to your Hugging Face account so you can consume the model directly from Cortex.

In [ ]:
# Authenticate to Hugging Face Hub
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import HfApi
import os

api = HfApi()

# TODO: Replace with your Hugging Face username
HF_USERNAME = "your-username"
REPO_NAME = "cortex-tuned-qwen"
repo_id = f"{HF_USERNAME}/{REPO_NAME}"

# Create repository on Hugging Face Hub
print(f"Creating repository {repo_id} on Hugging Face...")
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

# Upload the checkpoints folder containing model weights
if os.path.exists("./checkpoints"):
    print(f"Uploading checkpoints to HF Hub...")
    api.upload_folder(
        folder_path="./checkpoints",
        repo_id=repo_id,
        repo_type="model"
    )
    print(f"Model successfully uploaded! You can access it at https://huggingface.co/{repo_id}")
else:
    print("Error: './checkpoints' folder not found. Check if the training run completed and saved weights.")